# 01 - Preprocessing

In [ ]:
import os
import sys
from pathlib import Path

# Set the base project directory
base_proj_dir = Path("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis")

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")
sys.path.append(str(base_proj_dir))

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import snapseed as snap
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import marsilea as ma
import marsilea.plotter as mp
from dynaconf import Dynaconf

# Load settings
settings = Dynaconf(
    settings_files=[base_proj_dir / 'settings.toml', base_proj_dir / '.secrets.toml'],
)

sc.settings.figdir = settings.ANALYSIS.figures_dir

In [ ]:
data_dir = Path(settings.IO.combined_data) / "all-sample" / "DGE_filtered"
anndata_dir = Path(settings.IO.anndata)

## Load data  & plate assignments

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_combined_processed.h5ad")

In [ ]:
plate_assignments = pd.read_csv("/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/tf_ko_panel_analysis/plate_assignment.csv")

plate_assignments = plate_assignments.rename(columns={"Sample Name": "sample", "TF": "condition", "Batch": "batch", "Round within Batch": "round_within_batch"})
plate_assignments["sample"] = plate_assignments["sample"].str.replace(".", "_").apply(lambda x: f"sample_{x}" if not any(c.isalpha() for c in x) else x).unique()

In [ ]:
metadata = adata.obs.reset_index().copy()
metadata = metadata.merge(plate_assignments, on="sample", how="left").set_index("bc_wells")
metadata["condition"] = metadata["condition"].apply(lambda x: x.lower().capitalize())

adata.obs = metadata.copy()

In [ ]:
# Count how many samples we have per condition
adata.obs.drop_duplicates(subset=["condition", "sample"]).groupby("condition").size().reset_index(name="num_samples")

In [ ]:
plate_assignments.query('condition == "Isl1"')

In [ ]:
adata.obs.query('condition == "Isl1"')[["sample", "condition", "batch", "round_within_batch"]].drop_duplicates()

In [ ]:
adata

In [ ]:
adata.obs.head()

## Plot data

In [ ]:
sc.pl.umap(adata, color="pca_leiden_2.0", wspace=0.4, frameon=False, legend_loc="on data", save="_tfatlas_pca_leiden_2.0.pdf", show=True)

## Show distribution of controls on UMAP

In [ ]:
# Now we itereate through each TF and plot the UMAP
adata_plot = adata[adata.obs["condition"] == "Control"].copy()

for sample in  adata_plot.obs["sample"].unique().tolist():
    adata_plot.obs["is_sample"] = (adata_plot.obs["sample"] == sample).astype("category")
    adata_plot.obs["is_sample"] = adata_plot.obs["is_sample"].cat.reorder_categories([False, True])
    adata_plot = adata_plot[adata_plot.obs["is_sample"].sort_values().index]

    sc.pl.umap(
        adata_plot,
        color="is_sample",
        palette=["lightgrey", "black"],
        title=sample,
        frameon=False,
        wspace=0.4,
        size=3,
        save=f"_tfatlas_{sample}_highlight.pdf",
        show=False,
    )

## Show distribution of each TF on UMAP

In [ ]:
# Now we itereate through each TF and plot the UMAP
conditions = adata.obs["condition"].unique().tolist()
for cond in conditions:
    if cond == "Control":
        continue
    adata_plot = adata[adata.obs["condition"].isin(["Control"] + [cond])].copy()
    adata_plot.obs["is_condition"] = (adata_plot.obs["condition"] == cond).astype("category")
    adata_plot.obs["is_condition"] = adata_plot.obs["is_condition"].cat.reorder_categories([False, True])
    adata_plot = adata_plot[adata_plot.obs["is_condition"].sort_values().index]

    sc.pl.umap(
        adata_plot,
        color="is_condition",
        palette=["lightgrey", "black"],
        title=cond,
        frameon=False,
        wspace=0.4,
        size=3,
        save=f"_tfatlas_{cond}_highlight.pdf",
        show=False,
    )

In [ ]:
adata

In [ ]:
adata.write(anndata_dir / "tf_ko_panel_atlas.h5ad")

## Annotate atlas

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_atlas.h5ad")

In [ ]:
adata = adata.raw.to_adata()

In [ ]:
adata.raw = adata.copy()
adata.layers["counts"] = adata.X.copy()

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
del adata.uns["log1p"]

In [ ]:
adata.write(anndata_dir / "tf_ko_panel_atlas_vis.h5ad")

In [ ]:
adata = sc.read(anndata_dir / "tf_ko_panel_combined_controls_processed_vis.h5ad")

## Plot cell type markers

In [ ]:
# Show dotplot of markers
sc.pl.dotplot(adata, var_names=["LGR5", "SMOC2", "OLFM4", "TLN2", "SI", "FABP2", "APOA4", "MKI67", "TOP2A", "BEST4", "NOTCH2",
                                "FCGBP", "MUC2", "DEFA5", "DEFA6", "AVIL", "KIT", "NEUROG3", "PROX1", "CHGA", "TPH1", "GHRL", "MLN", "SST", "HHEX", 
                                "GIP", "RFX6", "NTS","CCK", "GCG", "PYY"], groupby="pca_leiden_2.0", standard_scale="var", cmap="Reds", swap_axes=True, dendrogram=True, figsize=(15,10), use_raw=False, show=False)

In [ ]:
features = [
    "LGR5", "SMOC2", "OLFM4", "TLN2", "SI", "FABP2", "APOA4", 
    "MKI67", "TOP2A", "BEST4", "NOTCH2", "FCGBP", "MUC2", 
    "DEFA5", "DEFA6", "AVIL", "KIT", "NEUROG3", "PROX1", 
    "CHGA", "TPH1", "GHRL", "MLN", "SST", "HHEX", 
    "GIP", "RFX6", "NTS","CCK", "GCG", "PYY"
]
for feature in features:
    sc.pl.umap(adata, color=feature, frameon=False, cmap="Purples", save=f"_tfatlas_{feature}_expression.pdf", show=False, use_raw=False)

## Preannotate using snapseed

In [ ]:
marker_dict = {
    "Epithelial": {
        "marker_genes": ["CDH1", "EPCAM", "KRT8"],  # generic epithelial
        "subtypes": {
            "Stem cells": {"marker_genes": ["LGR5", "SMOC2"]},
            "TA cells": {"marker_genes": ["OLFM4", "TLN2"]},
            "Enterocytes": {"marker_genes": ["SI", "FABP2", "APOA4"]},
            "Cycling cells": {"marker_genes": ["MKI67", "TOP2A"]},
            "BEST4+ enterocytes": {"marker_genes": ["BEST4", "NOTCH2"]},
            "Goblet cells": {"marker_genes": ["FCGBP", "MUC2"]},
            "Paneth cells": {"marker_genes": ["DEFA5", "DEFA6"]},
            "Tuft cells": {"marker_genes": ["AVIL", "KIT"]},
            "Enteroendocrine cells": {
                "marker_genes": ["CHGA", "NEUROG3", "PROX1", "RFX6"],
                "subtypes": {
                    "EC cells": {"marker_genes": ["TPH1"]},
                    "D cells": {"marker_genes": ["SST", "HHEX"]},
                    "X cells": {"marker_genes": ["GHRL", "MLN"]},
                    "K cells": {"marker_genes": ["GIP"]},
                    "L cells": {"marker_genes": ["GCG", "PYY"]},
                    "I cells": {"marker_genes": ["CCK"]},
                    "N cells": {"marker_genes": ["NTS"]},
                },
            },
        },
    },
}


In [ ]:
adata.obs["_cluster"] = adata.obs["pca_leiden_2.0"].copy()

In [ ]:
group_name = "_cluster"
res = snap.annotate_hierarchy(
    adata,
    marker_dict,
    group_name=group_name,
)

In [ ]:
adata_annotated = adata.copy()

for level in res["metrics"].keys():
    
    df = res["metrics"][level].copy()
    df.columns = [
        group_name if i == 0 else f"{level}_{col}"
        for i, col in enumerate(df.columns)
    ]
    
    adata_annotated.obs = adata_annotated.obs.merge(df, left_on=group_name, right_on=group_name, how="left")
    adata_annotated.obs_names = list(adata.obs_names)
    
adata_annotated.obs["cell_type"] = (
    adata_annotated.obs[["level_3_class", "level_2_class", "level_1_class"]]
    .bfill(axis=1)
    .iloc[:, 0]
)

In [ ]:
sc.pl.umap(adata_annotated, color=["pca_leiden_2.0", "cell_type"], wspace=0.4, frameon=False)

In [ ]:
# Show dotplot of markers
sc.pl.dotplot(adata_annotated, var_names=["LGR5", "SMOC2", "OLFM4", "TLN2", "SI", "FABP2", "APOA4", "MKI67", "TOP2A", "BEST4", "NOTCH2",
                                "FCGBP", "MUC2", "DEFA5", "DEFA6", "AVIL", "KIT", "NEUROG3", "PROX1", "CHGA", "TPH1", "GHRL", "MLN", "SST", "HHEX", 
                                "GIP", "RFX6", "NTS","CCK", "GCG", "PYY"], groupby="cell_type", standard_scale="var", cmap="Reds", swap_axes=True, dendrogram=True, figsize=(15,10), use_raw=False, show=False)

In [ ]:
adata_annotated

In [ ]:
del adata_annotated.uns["log1p"]

In [ ]:
from scipy.sparse import csc_matrix

if not isinstance(adata_annotated.X, csc_matrix):
    adata_annotated.X = csc_matrix( adata_annotated.X)

In [ ]:
adata_annotated.write(anndata_dir / "tf_ko_panel_combined_annotated.h5ad")

In [ ]:
anndata_dir / "tf_ko_panel_combined_annotated.h5ad"

## Show markers for each cluster

In [ ]:
markers = pd.read_csv(Path(settings.ANALYSIS.tables_dir) / "markers.csv", index_col=0)

In [ ]:
markers = markers.merge(adata.var.loc[keep_genes].reset_index().query("gene_name != gene_id"), left_on="names", right_on="gene_name", how="inner")

In [ ]:
markers.shape

In [ ]:
top_markers = (
    markers[["gene_name", "scores", "group"]]
    .sort_values(["group", "scores"], ascending=[True, False])
    .groupby("group")
    .head(20)
)

In [ ]:
top_markers.shape

In [ ]:
top_markers = list(markers["gene_name"].unique())

In [ ]:
len(top_markers)

In [ ]:
# Show dotplot of markers
sc.pl.dotplot(adata, var_names=top_markers, groupby="pca_leiden_2.0", standard_scale="var", cmap="Reds", swap_axes=True, dendrogram=True, figsize=(15,35), use_raw=False, show=False)

## Plot distribution of clusters by condition

In [ ]:
cells_per_sample_and_cluster = pd.crosstab(adata.obs["sample"], adata.obs["pca_leiden_2.0"], normalize=True)

In [ ]:
cells_per_sample_and_cluster

In [ ]:
pd.cells_per_sample_and_cluster.reset_index()